# 사후 검증 시뮬레이션 — A(감각) vs B(모델) vs C(일평균)

`04_integrated_model_loocv.ipynb`에서 만든 T2 모델(`금` + `roll_prev_3days_mean_t2`)을, 학습에 전혀 쓰지 않은 새 날짜(n=4, 2026-07-03~07-10)에 적용해 세 방식의 오차를 비교한다.

- **A**: 사장님 감각 기반 초벌 투입량(10kg, 수분손실 수율 88% 반영) → 실제 소화량 오차
- **B**: T2 모델 예측값 → 실제 소화량 오차
- **C**: T2 타겟 전체 평균(8,230g) 고정값 → 실제 소화량 오차


## 저장된 모델 불러오기


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

final_model_t2 = sm.load('/content/model_t2_v1.pickle')
print(final_model_t2.summary())


                            OLS Regression Results                            
Dep. Variable:            cumul_20_22   R-squared:                       0.505
Model:                            OLS   Adj. R-squared:                  0.434
Method:                 Least Squares   F-statistic:                     7.138
Date:                Tue, 14 Jul 2026   Prob (F-statistic):            0.00730
Time:                        05:15:46   Log-Likelihood:                -152.80
No. Observations:                  17   AIC:                             311.6
Df Residuals:                      14   BIC:                             314.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                    1

## 검증 데이터셋 불러오기


In [2]:
valid_raw = pd.read_csv('/content/vaild_data_set.csv')  # 원본 보존용 (dcl_id, date, 실제값 등 포함)
valid_raw.head()


,dcl_id,date,day_of_week,cumul_20_22,roll_prev_3days_mean_t2
0,dcl_0703,2026-07-03,금,4850,7483.33
1,dcl_0706,2026-07-06,월,6200,5833.33
2,dcl_0707,2026-07-07,화,11150,6583.33
3,dcl_0710,2026-07-10,금,11200,7400.00


## 모델 입력 형태로 변환

원본(`valid_raw`)은 그대로 두고, 모델 입력용 피처만 별도 데이터프레임(`X_input`)으로 만든다 — 원본을 덮어쓰면 나중에 날짜·실제값 컬럼에 접근할 때 헷갈리기 쉽다.


In [3]:
X_input = pd.DataFrame({
    '금': (valid_raw['day_of_week'] == '금').astype(int),
    'roll_prev_3days_mean_t2': valid_raw['roll_prev_3days_mean_t2'],
})
X_input = sm.add_constant(X_input, has_constant='add')
print(X_input)

prediction = final_model_t2.predict(X_input)
print(prediction)


   const  금  roll_prev_3days_mean_t2
0    1.0  1                  7483.33
1    1.0  0                  5833.33
2    1.0  0                  6583.33
3    1.0  1                  7400.00
0    9767.935031
1    7226.353468
2    6738.219978
3    9822.169916
dtype: float64


## A/B/C 방식별 오차(잔차) 계산

- **A안**: 사장님이 감각으로 초벌 10kg을 투입한다고 가정하고, 수분손실 수율(0.88)을 반영한 실제 소진 가능량과 비교
- **B안**: 모델 예측값과 실제값의 차이
- **C안**: T2 타겟 전체 평균(8,230g, H1 가설검증 단계에서 확인한 일평균 기준)과 실제값의 차이


In [4]:
y_true = valid_raw['cumul_20_22']

resid_A = (10 * 0.88) * 1000 - y_true          # 감각(10kg) × 수율 88% 반영
resid_B = prediction - y_true                   # 모델 예측
resid_C = 8230 - y_true                         # T2 평균 고정값


## 결과 테이블 구성


In [5]:
result = valid_raw.copy()
result['prediction'] = prediction
result['A_residual'] = resid_A
result['B_residual'] = round(resid_B, 0)
result['C_residual'] = resid_C
result


,dcl_id,date,day_of_week,cumul_20_22,roll_prev_3days_mean_t2,prediction,A_residual,B_residual,C_residual
0,dcl_0703,2026-07-03,금,4850,7483.33,9767.935031,3950.0,4918.0,3380
1,dcl_0706,2026-07-06,월,6200,5833.33,7226.353468,2600.0,1026.0,2030
2,dcl_0707,2026-07-07,화,11150,6583.33,6738.219978,-2350.0,-4412.0,-2920
3,dcl_0710,2026-07-10,금,11200,7400.00,9822.169916,-2400.0,-1378.0,-2970


## 승패 비교 — B(모델)가 A·C를 이기는가


In [6]:
result['A_abs'] = result['A_residual'].abs()
result['B_abs'] = result['B_residual'].abs()
result['C_abs'] = result['C_residual'].abs()

result['B_beats_A'] = result['B_abs'] < result['A_abs']
result['B_beats_C'] = result['B_abs'] < result['C_abs']
result['B_wins_both'] = result['B_beats_A'] & result['B_beats_C']

print(f"B가 A를 이긴 비율: {result['B_beats_A'].mean():.0%}")
print(f"B가 C를 이긴 비율: {result['B_beats_C'].mean():.0%}")
print(f"B가 둘 다 이긴 비율: {result['B_wins_both'].mean():.0%}")
print()
print(f"전체 MAE - A: {result['A_abs'].mean():.1f} / B: {result['B_abs'].mean():.1f} / C: {result['C_abs'].mean():.1f}")

hit = result[result['B_wins_both']]
miss = result[~result['B_wins_both']]


B가 A를 이긴 비율: 50%
B가 C를 이긴 비율: 50%
B가 둘 다 이긴 비율: 50%

전체 MAE - A: 2825.0 / B: 2933.5 / C: 2825.0


## 승/패 조건별 MAE 비교


In [7]:
print(f"예측 성공일 MAE - A: {hit['A_abs'].mean():.1f} / B: {hit['B_abs'].mean():.1f} / C: {hit['C_abs'].mean():.1f}")
print(f"예측 실패일 MAE - A: {miss['A_abs'].mean():.1f} / B: {miss['B_abs'].mean():.1f} / C: {miss['C_abs'].mean():.1f}")


예측 성공일 MAE - A: 2500.0 / B: 1202.0 / C: 2500.0
예측 실패일 MAE - A: 3150.0 / B: 4665.0 / C: 3150.0


In [8]:
success_rate = (hit['A_abs'].mean() - hit['B_abs'].mean()) / hit['A_abs'].mean() * 100
failure_rate = abs((miss['A_abs'].mean() - miss['B_abs'].mean()) / miss['A_abs'].mean() * 100)

print(f"예측 성공일 오차 변화율: {success_rate:.2f}% 감소")
print(f"예측 실패일 오차 변화율: {failure_rate:.2f}% 증가")


예측 성공일 오차 변화율: 51.92% 감소
예측 실패일 오차 변화율: 48.10% 증가


## roll_prev 괴리 확인 (승패를 가르는 조건 참고용)

`roll_prev`(피처)와 실제값의 차이가 클수록 예측이 크게 빗나갈 것 같지만, 실제로는 두 요일 그룹(평일/금요일)의 절편이 그날 실제 수준에 얼마나 가까운지가 더 중요했다(자세한 해석은 노션 문서 참고).


In [9]:
result['roll_prev_3days_mean_t2'] - result['cumul_20_22']


,0
0,2633.33
1,-366.67
2,-4566.67
3,-3800.00


---
## 종합

- **전체 MAE**: A 2,825.0 / B 2,933.5 / C 2,825.0 — 전체 평균만 보면 B(모델)가 A·C와 비슷하거나 소폭 열세
- **승패로 나누면**: 예측 성공일(2일) 오차 51.9% 감소, 예측 실패일(2일) 오차 48.1% 증가 — 두 변화율이 비슷해 "이길 때 압도적으로 잘 맞고 질 때는 덜 틀린다"는 식의 비대칭성 근거는 약하다
- **한계**: n=4로 승/패 그룹이 각 2개뿐이라, 이 조건부 패턴이 구조적 신호인지 우연인지는 아직 판단할 수 없다. 표본이 쌓이는 대로 재검증이 필요하다.
